Transformers Would not load for me if I didn't uninstall and reinstall

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Pip install transformers

In [ ]:
!pip install transformers

Loading the model directly using the HF api token.

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

import os
from huggingface_hub import InferenceClient
os.environ["HF_TOKEN"] = "API_Key"
client = InferenceClient(
    provider="auto",
    api_key=os.environ["HF_TOKEN"],)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4.1 Sentiment State Extraction API key and Tokenizer

In [48]:
text_example = ["Apple stock rose after strong earnings","Guidance disappointed investors"]

results=[]
for text in text_example:
    r = client.text_classification(
        text,
        model="ProsusAI/finbert",
        top_k=3
    )
    results.append(r)

s_i_scores = []
HD_i_vector =[]

# Si(t) is [1,0,-1]*H(Di(t)). Loop keeps both H(Di(t)) and Si(t)
for r in results:
  positive = r[0]["score"]
  neutral = r[1]["score"]
  negative = r[2]["score"]

  score = positive-negative
  HD_i_vector.append([positive,neutral,negative])
  s_i_scores.append(score)


print(HD_i_vector,"\n",s_i_scores)

[[0.9266610145568848, 0.04188281670212746, 0.031456105411052704], [0.864722728729248, 0.06892205029726028, 0.06635522842407227]] 
 [0.8952049091458321, 0.7983675003051758]


4.1 Using the Tokenizer and API key

In [43]:
# Use a pipeline as a high-level helper
from transformers import pipeline
pipe = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)  #top_k=None give pos-nue-neg values


text_example = ["Apple stock rose after strong earnings","Guidance disappointed investors"]


results = pipe(text_example)


s_i_scores = []
HD_i_vector =[]


# Si(t) is [1,0,-1]*H(Di(t)). Loop keeps both H(Di(t)) and Si(t)
for r in results:
 positive = r[0]["score"]
 neutral = r[1]["score"]
 negative = r[2]["score"]


 score = positive-negative
 HD_i_vector.append([positive,neutral,negative])
 s_i_scores.append(score)

print(HD_i_vector,"\n",s_i_scores)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[0.9266611337661743, 0.04188275337219238, 0.03145609796047211], [0.8647229075431824, 0.06892195343971252, 0.0663551315665245]] 
 [0.8952050358057022, 0.7983677759766579]
